In [83]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
from pyhessian import hessian
from tqdm import tqdm
import pandas as pd
import os

device = "cuda"

# export visible cuda to only use GPU 7
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

torch.set_printoptions(precision=5, sci_mode=False)

def print_grads_norm(model, grads):
    for n, g in zip(model.state_dict().keys(), grads):
        if g is None:
            print(f'{n:20s}  ← NOT in graph')
        else:
            print(f'{n:20s}  ‖g‖₂ = {g.norm().item():.6e}   max|g| = {g.abs().max().item():.6e}')

In [65]:
torch.manual_seed(0)
np.random.seed(0)

A = np.random.rand(10, 10)
print("Top 3 eivenvalues of A: ", np.linalg.eigvals(A)[0:3])

Top 3 eivenvalues of A:  [4.82560951+0.j 0.96051159+0.j 0.82240406+0.j]


In [66]:
# approximate the top eigenvalue using power iteration
v = np.random.rand(10, 1)
for i in range(100):
    v = A @ v
    v = v / np.linalg.norm(v) # numerical stability

# Rayleigh quotient
top_eigenvalue = (v.T @ A @ v) / (v.T @ v)
print("Top eigenvalue with Rayleigh quotient: ", top_eigenvalue.flatten())

Top eigenvalue with Rayleigh quotient:  [4.82560951]


In [67]:
# we can find subsequent eigenvalues by deflating the matrix
# deflating works because A = sum_i (lambda_i * v_i @ v_i^T) (under specific conditions)
A2 = A - top_eigenvalue * (v @ v.T)

v = np.random.rand(10, 1)
for i in range(200):
    v = A2 @ v
    v = v / np.linalg.norm(v) # numerical stability

# Rayleigh quotient
top_eigenvalue2 = (v.T @ A2 @ v) / (v.T @ v)
print("Top eigenvalue of A2 with Rayleigh quotient: ", top_eigenvalue2.flatten())

Top eigenvalue of A2 with Rayleigh quotient:  [0.96051159]


In [68]:
from train_mlp import muMLPTab9, SP_MLP, NTK_MLP, train, preload_subset

dl = preload_subset(32, 0.1)

sharp_iter = iter(dl)
# sharp_x1, sharp_y1 = next(sharp_iter)
# sharp_x2, sharp_y2 = next(sharp_iter)
# sharp_x, sharp_y = torch.cat([sharp_x1, sharp_x2]), torch.cat([sharp_y1, sharp_y2]).to(device)
sharp_x, sharp_y = next(sharp_iter)

In [84]:
# model = SP_MLP(128).to(device)
model = muMLPTab9(16).to(device)
lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0)
num_epochs = 1
train_dl = dl
criterion = nn.CrossEntropyLoss()

lambdas = []
losses = []

model.train()
for epoch in tqdm(range(num_epochs)):
    train_loss = 0
    for batch_idx, (data, target) in enumerate(train_dl):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        train_loss += loss.item() * data.size(0)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    # compute the Hessian
    model.eval()
    hess = hessian(model, criterion=criterion, data=(sharp_x, sharp_y), cuda=True)
    lambdas.append(hess.eigenvalues(top_n=10)[0])
    losses.append(train_loss / len(train_dl.dataset))

    print(train_loss / len(train_dl.dataset))

100%|██████████| 1/1 [00:01<00:00,  1.01s/it]

2.6916294773101805


In [ ]:
torch.manual_seed(0)

x, y = sharp_x.to(device), sharp_y.to(device)
loss = criterion(model(x), y)
print("Loss: ", loss)

# first order gradients of the loss w.r.t. the model parameters
grads = torch.autograd.grad(loss, model.parameters(), retain_graph=True, create_graph=True)

for i in range(len(grads)):
    print(grads[i].shape)
print_grads_norm(model, grads)

# random vectors to compute the Hessian-vector product
v = [torch.randn(p.size()).to(device) for p in model.parameters()]

for i in range(len(v)):
    v[i] = v[i] / torch.norm(v[i]) # numerical stability
    print(v[i].shape)

gv = sum((g * vv).sum() for g, vv in zip(grads, v))
print("gv: ", gv)

hvp = torch.autograd.grad(gv, model.parameters(), retain_graph=True)

print("Hessian-vector product: ")
for i in range(len(hvp)):
    print(hvp[i].shape)

Loss:  tensor(2.41295, device='cuda:0', grad_fn=<NllLossBackward0>)
torch.Size([16, 3072])
torch.Size([16, 16])
torch.Size([10, 16])
fc_1.weight           ‖g‖₂ = 9.138839e-01   max|g| = 2.142279e-02
fc_2.weight           ‖g‖₂ = 8.675670e-01   max|g| = 2.398877e-01
fc_3.weight           ‖g‖₂ = 8.749942e-01   max|g| = 3.037695e-01
torch.Size([16, 3072])
torch.Size([16, 16])
torch.Size([10, 16])
gv:  tensor(-0.06698, device='cuda:0', grad_fn=<AddBackward0>)
Hessian-vector product: 
torch.Size([16, 3072])
torch.Size([16, 16])
torch.Size([10, 16])
